# Figure S3 — CC-Perturb-seq data processing and quality control

Second-pass implementation of panels a–f. The preprocessing input may be a standardized QC directory or an AnnData file; barcode-rank and pre-QC panels require the corresponding unfiltered tables.


In [ ]:
import matplotlib.pyplot as plt
from src.cc_perturbseq.figure_notebook import get_path, initialize_figure, load_figure_manifest, panel_directory, write_receipt
from src.cc_perturbseq.program_panels import *

def export(fig, panel, stem):
    out = panel_directory(ctx, panel)
    result = {}
    for suffix, kwargs in [("pdf", {}), ("svg", {}), ("png", {"dpi": 300})]:
        path = out / f"{stem}.{suffix}"
        fig.savefig(path, **kwargs)
        result[f"{stem}_{suffix}"] = str(path)
    return result

FIGURE = load_figure_manifest("Figure_S03")
ctx = initialize_figure(FIGURE)
ctx["input_audit"]


## Panels a–e — sequencing and guide-capture QC


In [ ]:
qc_path = get_path(ctx["paths"], "ccperturbseq.preprocessing_qc")
bundle = load_qc_bundle(qc_path)
all_outputs = {}
all_tables = {}
if "barcode_rank" in bundle:
    fig_a, source_a = plot_barcode_rank(bundle["barcode_rank"])
    all_outputs.update(export(fig_a, "a", "Figure_S03a_barcode_rank"))
    all_tables.update(save_source_tables({"barcode_rank": source_a}, panel_directory(ctx, "a"), "Figure_S03a"))
fig_b, source_b = plot_umi_gene_scatter(bundle)
all_outputs.update(export(fig_b, "b", "Figure_S03b_umi_gene_scatter"))
all_tables.update(save_source_tables({"cell_metrics": source_b}, panel_directory(ctx, "b"), "Figure_S03b"))
fig_c, source_c = plot_metric_distributions(bundle)
all_outputs.update(export(fig_c, "c", "Figure_S03c_qc_distributions"))
all_tables.update(save_source_tables({"cell_metrics": source_c}, panel_directory(ctx, "c"), "Figure_S03c"))
fig_d, source_d = plot_cells_retained(bundle)
all_outputs.update(export(fig_d, "d", "Figure_S03d_cells_retained"))
all_tables.update(save_source_tables({"cells_retained": source_d}, panel_directory(ctx, "d"), "Figure_S03d"))
fig_e, source_e = plot_guide_metrics(bundle)
all_outputs.update(export(fig_e, "e", "Figure_S03e_guide_metrics"))
all_tables.update(save_source_tables({"guide_metrics": source_e}, panel_directory(ctx, "e"), "Figure_S03e"))
for panel in ["a","b","c","d","e"]:
    if panel != "a" or "barcode_rank" in bundle:
        write_receipt(ctx, panel, status="IMPLEMENTED_NOT_YET_NUMERICALLY_VALIDATED", inputs={"preprocessing_qc": str(qc_path)}, outputs={key:value for key,value in {**all_outputs, **all_tables}.items() if f"S03{panel}" in key or f"panel_{panel}" in value}, parameters={})
plt.show()


## Panel f — on-target knockdown efficiency


In [ ]:
kd_path = get_path(ctx["paths"], "sceptre.on_target_results")
knockdown = standardize_knockdown_results(kd_path)
fig_f, source_f = plot_knockdown_distribution_by_day(knockdown)
outputs = export(fig_f, "f", "Figure_S03f_target_knockdown_by_day")
tables = save_source_tables({"knockdown": source_f}, panel_directory(ctx, "f"), "Figure_S03f")
write_receipt(ctx, "f", status="IMPLEMENTED_NOT_YET_NUMERICALLY_VALIDATED", inputs={"sceptre_results": str(kd_path)}, outputs={**outputs, **tables}, parameters={"effect": "100 × (1 - SCEPTRE fold change)", "significance": "BH q <= 0.05 within day", "point_unit": "target gene"})
plt.show()
